# Synthetic Tabular Data Generation - Streamlined Driver

**Streamlined notebook for synthetic tabular data generation with batch processing.**

This notebook provides a config-driven workflow for:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: Hyperparameter optimization
- Section 5: Final models with best parameters

Based on STG-BreastCancer-testing.ipynb, streamlined for faster iteration.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)
import sys                                                                                                                                                                                                     
!{sys.executable} -m pip install ctgan sdv ganeraid 

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[CONFIG] Session timestamp: 2026-01-30
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementations
[OK] Config-driven preprocessing functions loaded (Phase 3)
[OK] EDA mod

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/liver_train.csv",  # Path to your CSV file
    "target_column": "Result",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Liver Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    "categorical_columns": ['gender_of_the_patient'],     # List categorical cols, or [] for auto-detect
    # 0 = Female, 1 = Male
    "task_type": "auto",                          # "auto" | "classification" | "regression"

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": True,                      # True to sample rows for faster testing
    "sample_n": 3000,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "indicator_onehot",       # "none" | "drop" | "median" | "mode" | "mice" | "indicator_onehot"
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    # "models_to_run": ["tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "medgan", "pategan"],     # "all" or list like ["ctgan", "tvae"]
    "models_to_run": "all",     # "all" or list like ["ctgan", "tvae"]
    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 5,                          # Trials for smoke testing
    "n_trials_full": 30,                          # Trials for full optimization
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/liver_train.csv
  Target column: Result
  Models to run: all
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

# Load and preprocess data using the config
(
    data,                  # Processed DataFrame
    original_data,         # Copy for reference
    target_column,         # Target column name (cleaned)
    DATASET_IDENTIFIER,    # Dataset identifier for results paths
    categorical_columns,   # List of categorical columns
    metadata               # Full preprocessing metadata
) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

# Set aliases for backward compatibility with existing notebook cells
TARGET_COLUMN = target_column

# Get models to run based on config
models_to_run = get_models_to_run(NOTEBOOK_CONFIG)
tuning_config = get_tuning_config(NOTEBOOK_CONFIG)
N_TRIALS = get_n_trials(NOTEBOOK_CONFIG)

# Set RUN_MODE for backward compatibility with Section 4 tuning cells
RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/liver_train.csv
[LOAD] Loaded 30691 rows, 11 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (30691, 11)
[PREPROCESS] Categorical columns: ['gender_of_the_patient']
[PREPROCESS] Task type: classification
[PREPROCESS] Added missingness indicator: age_of_the_patient__is_missing
[PREPROCESS] Added missingness indicator: gender_of_the_patient__is_missing
[PREPROCESS] Added missingness indicator: total_bilirubin__is_missing
[PREPROCESS] Added missingness indicator: direct_bilirubin__is_missing
[PREPROCESS] Added missingness indicator: alkphos_alkaline_phosphotase__is_missing
[PREPROCESS] Added missingness indicator: sgpt_alamine_aminotransferase__is_missing
[PREPROCESS] Added missingness indicator: sgot_aspartate_aminotransferase__is_missing
[PREPROCESS] Added missingness indicator: total_protiens__is_missing
[PREPROCESS] Added missingness indicator: alb_albumin__is_missing
[PREPROCESS] Added missingness indicator: a/g_ratio_albu

### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# Generates: column_analysis.csv, target_analysis.csv, target_balance_metrics.csv,
#            feature_distributions.png, correlation_heatmap.png, correlation_matrix.csv
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Liver Dataset
Target column: result
Results path: results/liver-train/2026-01-30/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Liver Dataset
   Shape......................... 3000 rows x 22 columns
   Memory Usage.................. 0.53 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 151
   Numeric Columns............... 22
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (22 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.397 (Highly Imbalanced)

[4/6] Feature Distributions...
   Saved: 4 distribution file(s) (21 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (21 features)

[6/6] Configuration Validation...
   Categorical columns: 

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
# This creates global variables like synthetic_data_ctgan, synthetic_data_tvae, etc.
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success':
        globals()[f'{model_name}_model'] = result['model']

CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Model ctgan failed: Invalid columns found: {'gender_of_the_patient'}



BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (3000, 22)
Target column: result
Samples to generate: 3000


[1/7] Training CTGAN...
--------------------------------------------------
  Training CTGAN...
  [ERROR] CTGAN failed: Invalid columns found: {'gender_of_the_patient'}

[2/7] Training TVAE...
--------------------------------------------------
  Training TVAE...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 3000 synthetic samples...


[COPULAGAN] Column 'age_of_the_patient__is_missing' has near-zero range (range=0.00000000, std=0.00000000)


[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 52.83s
  Synthetic data shape: (3000, 22)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Training CopulaGAN...
  Generating 3000 synthetic samples...
  [OK] CopulaGAN completed in 94.29s
  Synthetic data shape: (3000, 22)

[4/7] Training CTABGAN...
--------------------------------------------------
  Training CTABGAN...


100%|██████████| 300/300 [02:38<00:00,  1.90it/s]


Finished training in 166.38492798805237  seconds.
  Generating 3000 synthetic samples...
  [OK] CTABGAN completed in 166.63s
  Synthetic data shape: (3000, 22)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Training CTABGAN+...


100%|██████████| 400/400 [04:57<00:00,  1.34it/s]


Finished training in 306.50897908210754  seconds.
  Generating 3000 synthetic samples...
  [OK] CTABGAN+ completed in 306.71s
  Synthetic data shape: (3000, 22)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Training PATE-GAN...
  Generating 3000 synthetic samples...
  [OK] PATE-GAN completed in 13.04s
  Synthetic data shape: (3000, 22)

[7/7] Training MEDGAN...
--------------------------------------------------
  Training MEDGAN...
  Generating 3000 synthetic samples...
  [OK] MEDGAN completed in 51.26s
  Synthetic data shape: (3000, 22)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 6
Failed: 1

Successful models:
  - TVAE: 52.83s
  - CopulaGAN: 94.29s
  - CTABGAN: 166.63s
  - CTABGAN+: 306.71s
  - PATE-GAN: 13.04s
  - MEDGAN: 51.26s

Failed models:
  - CTGAN: Invalid columns found: {'gender_of_the_patient'}

Created variables: ['synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),
    models_to_evaluate=None,
    real_data=None,
    target_col=None
)

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: liver-train
[INFO] Target column: result
[INFO] Variable pattern: standard
[INFO] Found 6 trained models:
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTABGAN ====================
[SEARCH] CTABGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/liver-train/2026-01-30/Section-3/CTABGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.787

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.012
   [CHART] Explained Variance (PC1, PC2): 0.128, 0.093

[3] DISTRIBUTION SIMILARITY
-----------------

## 4 Hyperparameter Tuning

### 4.1 Batch Hyperparameter Optimization

In [ ]:
# Code Chunk ID: CHUNK_4_1_BATCH
# ============================================================================
# SECTION 4.1 - BATCH HYPERPARAMETER OPTIMIZATION
# ============================================================================

import sys
import importlib

# Ensure Optuna is installed
!{sys.executable} -m pip install -U optuna -q

import optuna
print(f"Optuna version: {optuna.__version__}")

# Reload modules to pick up optuna
import src.models.search_spaces as ss
ss = importlib.reload(ss)

import src.models.batch_optimization as bo
bo = importlib.reload(bo)

optimize_models_batch = bo.optimize_models_batch
extract_studies_to_globals = bo.extract_studies_to_globals

# Run batch optimization
optimization_results = optimize_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_trials=N_TRIALS,
    run_mode=RUN_MODE,
    random_state=42,
    verbose=False
)

# Extract *_study variables for Section 4.2 compatibility
created_vars = extract_studies_to_globals(optimization_results, globals())
print(f"\nCreated study variables: {created_vars}")

[I 2026-01-30 19:15:43,230] A new study created in memory with name: ctgan_hpo


Optuna version: 4.7.0
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py


CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 0 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 1 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 2 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 3 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 5 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 6 for ctgan failed: Invalid columns found: {'gender_of_the_patient'}
CTGAN training failed: Invalid columns found: {'gender_of_the_patient'}
Trial 7 for ctgan failed: Invalid columns foun

[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7760
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7567
[CHART] Combined Score: 0.7683 (Similarity: 0.7760, Accuracy: 0.7567)
[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7793
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7747
[CHART] Combined Score: 0.7774 (Similarity: 0.7793, Accuracy: 0.7747)
[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7820
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7750
[CHART] Combined Score: 0.7792 (Similarity: 0.7820, Accuracy: 0.7750)
[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7870
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7502
[CHART] Combined Score: 0.7722 (Similarity: 0.7870, Accuracy: 0.75

[COPULAGAN] Column 'age_of_the_patient__is_missing' has near-zero range (range=0.00000000, std=0.00000000)


...

100%|██████████| 150/150 [06:13<00:00,  2.49s/it]


Finished training in 381.4677212238312  seconds.
[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7664
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6683
[CHART] Combined Score: 0.7272 (Similarity: 0.7664, Accuracy: 0.6683)


100%|██████████| 150/150 [02:20<00:00,  1.07it/s]


Finished training in 148.30887961387634  seconds.
[TARGET] Enhanced objective function using target column: 'result'
[OK] Similarity Analysis: 22/22 valid metrics, Average: 0.7535
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5232
[CHART] Combined Score: 0.6614 (Similarity: 0.7535, Accuracy: 0.5232)


 10%|▉         | 68/700 [03:13<30:32,  2.90s/it]

### 4.2 Save Best Parameters

In [ ]:
# Code Chunk ID: CHUNK_4_1_SAVE
# ============================================================================
# SECTION 4.2 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

### 4.3 Hyperparameter Optimization Batch Analysis

In [ ]:
import inspect
print(inspect.signature(evaluate_hyperparameter_optimization_results))

In [ ]:
# Code Chunk ID: CHUNK_4_2_0_A
# ============================================================================
# SECTION 4.3 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
# ============================================================================

print("SECTION 4.3 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS")
print("=" * 80)

# Build scope dict for evaluation
study_vars_found = sorted([k for k in globals().keys() if k.endswith("_study")])
print(f"Study vars found: {study_vars_found}")

scope_for_eval = dict(globals())

# Use batch evaluation function
try:
    section4_batch_results = evaluate_hyperparameter_optimization_results(
    section_number=4,
    scope=scope_for_eval,
    target_column=TARGET_COLUMN)
    
    print(f"\nSECTION 4 BATCH ANALYSIS COMPLETED!")
    print(f"Models analyzed: {section4_batch_results.get('models_analyzed', 0)}")
    
except Exception as e:
    print(f"Section 4 batch analysis error: {e}")
    import traceback
    traceback.print_exc()

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [ ]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# Replaces individual 5.1.1-5.1.6 model cells with single batch call
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4
# This creates synthetic_*_final variables automatically
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,  # Load params from Section 4
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),   # Creates synthetic_*_final variables
    verbose=True
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")

### 5.2 Batch Evaluation of Optimized Models

In [ ]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

try:
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
    
    # Show summary
    if 'evaluation_summaries' in section5_batch_results:
        print("\nEVALUATION SUMMARIES:")
        print("-" * 40)
        for model_name, summary in section5_batch_results['evaluation_summaries'].items():
            print(f"{model_name}:")
            print(f"   Synthetic samples: {summary.get('synthetic_samples', 'N/A')}")
            print(f"   Overall score: {summary.get('overall_score', 'N/A')}")
            
except Exception as e:
    print(f"Section 5.2 batch evaluation failed: {e}")
    import traceback
    traceback.print_exc()

### 5.3 Final Summary

In [ ]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)